# Verify guide position for each sub/pose

In [11]:
from pathlib import Path
import pyvista as pv
import numpy as np
import json

from phd_helpers.paths import transform_mesh, get_relative_transform_new_basis, get_subject_stl_path, get_bone_transforms, pose2idCMC, get_bone_inertia, get_boundary
from phd_helpers.AbaqusPreprocessing import position_mc1_tpm

In [108]:
# load sub, pose, guide_id
params = []
with open("params.jsonl", "r") as f:
    for line in f:
        param = json.loads(line)
        params.append(param)
print(len(params), 'joints')
params[:1]

15 joints


[{'sub': '50017L',
  'pose': 'pinch_load',
  't': [0, 3.5, 1.5],
  'R': [0, 0, -15],
  'guide_id': 1}]

In [ ]:
i = 9

param = params[i]
sub = param['sub']
pose = param['pose']
guide_id = param['guide_id']
print(sub, pose, param['t'], param['R'])

# load bone meshes
mesh_dir = Path('../../../3Dprints/instronJigPipeline/outputs/CA-35T/meshes')
mc1_mesh_paths = list(mesh_dir.glob(f'{sub}/mc1_*[ge]-35T.vtp'))
tpm_mesh_paths = list(mesh_dir.glob(f'{sub}/*{pose}*.vtp'))
mc1_mesh = pv.read(mc1_mesh_paths[0]) + pv.read(mc1_mesh_paths[1])
tpm_mesh = pv.read(tpm_mesh_paths[0]) + pv.read(tpm_mesh_paths[1])
mc1_mesh['region_id'] = np.full(mc1_mesh.n_points, -2)
tpm_mesh['region_id'] = np.full(tpm_mesh.n_points, -2)
#tpm_mesh.points[:, 0] -= 2 # set around 2 mm from mc1
position_mc1_tpm(mc1_mesh, tpm_mesh, 2)

# load guide mesh
guide_mesh = pv.read(f'guide_meshes/guide_{guide_id}.stl')
cart_legs_offset = 17.5 # distance of lower surface of guide legs from peak of mc1 cartilage
legs_lower = guide_mesh.points[:, 0].max()
cart_peak = mc1_mesh.points[:, 0].min()
diff = abs(legs_lower - cart_peak) - cart_legs_offset
guide_mesh.points[:, 0] -= diff - 0.5 # don't know why just diff doesn't work (also slightly different for each sub)
if sub == '50034R':
    guide_mesh.points[:, 2] += 2 # add spacer for big one

pl = pv.Plotter()
pl.add_mesh(guide_mesh, color='gray')
pl.add_mesh(tpm_mesh, color='white')
pl.add_mesh(mc1_mesh, color='white')
pl.camera_position = pv.CameraPosition(position=(-28.07018759489931, -2.3535064429309576, -55.36319277262723),
               focal_point=(-18.950626850128174, -2.5172319412231445, 1.75),
               viewup=(-0.98733233969233, -0.01836126151094428, 0.15759985746629465))
pl.show()

50000R abduction [0, -2.5, 1.5] [0, 0, 15]
Distance between cartilage surfaces (x->): 2.0000
No interference:  True


Widget(value='<iframe src="http://localhost:55457/index.html?ui=P_0x3f97d3530_91&reconnect=auto" class="pyvist…